## 🔄 Restore Table en Delta Lake

Hasta ahora hemos visto cómo **Time Travel** nos permite consultar versiones anteriores de una Delta Table.

Pero Delta Lake va un paso más allá. Y, además de consultar el pasado, también podemos **restaurar una tabla completa a un estado anterior**, utilizando la información almacenada en el Transaction Log.

La restauración se basa en 2 enfoques:
* Restauración por Versión
* Restauración por Fecha y Hora específica. 

Esta capacidad resulta especialmente útil cuando:

* ✅ Se ejecuta una actualización incorrecta
* ✅ Se eliminan registros por error
* ✅ Un proceso ETL genera resultados inesperados
* ✅ Es necesario recuperar un estado previo de la información

---

## 🤯 ¿Qué ocurre internamente?

A diferencia de otros sistemas donde una restauración sobrescribe el historial existente, Delta Lake conserva la trazabilidad completa de la operación.

Al ejecutar un `RESTORE`, Databricks:

* 📜 Consulta el historial transaccional
* ⏳ Identifica la versión objetivo
* 🔄 Reconstruye el estado de la tabla
* 📝 Registra la restauración como una nueva operación

Como resultado, el historial permanece intacto y se genera una **nueva versión dentro de la Delta Table**, manteniendo la capacidad de auditoría y seguimiento de cambios.


### 🚀 Punto de Inicio en Databricks

Antes de trabajar con Delta Lake necesitamos una sesión de Spark activa.

Spark será el motor encargado de:

* ✅ Leer datos
* ✅ Transformarlos
* ✅ Procesarlos de forma distribuida
* ✅ Persistirlos como Delta Tables

In [0]:
from pyspark.sql import SparkSession # Puerta de entrada para trabajar con spark <-- SIEMPRE DEBEMOS IMPORTAR LA LLAVE MAESTRA QUE INICIA TODO.
from pyspark.sql.functions import *  # Funciones propias del módulo SQL de Spark, para trabajar sobre Dataframes.
spark = SparkSession.builder.appName("03RestoreTable").getOrCreate() 
"""
^          ^__________^        ^_________^                               ^
|                |                   |                                   | 
Variable   Constructor de Sesión   Nombre Aplicación       Evita conflicto del SparkSession"""

print("🚀 Spark Session iniciada correctamente")

#### 🔍 ¿Qué acaba de ocurrir?

Acabamos de crear una Spark Session. Esta sesión representa nuestro punto de entrada hacia:
* 🏗️ Apache Spark
* 🏗️ Databricks Runtime
* 🏗️ Delta Lake
* 🏗️ Unity Catalog

### ⚡ Utilizaremos la Delta Table del notebook anterior

- Nombre: `customers_delta_table`

In [0]:
## LECTURA INICIAL DE SU INFORMACIÓN
# display(spark.sql("SELECT * FROM catalog_databricks_2026_de.schema_databricks_2026_de.customers_delta_table"))
### ➡️ Resultado: 3 registros

##================================================

## REVISAMOS SU HISTORIAL
display(spark.sql("DESCRIBE HISTORY catalog_databricks_2026_de.schema_databricks_2026_de.customers_delta_table"))
### ➡️ Resultado: 3 Versiones


### 🔢 Restauración por Versión

Permite restaurar la tabla a una versión específica registrada en su historial.

```sql
RESTORE TABLE catalog.schema.table
TO VERSION AS OF numero_version;
```

🎯 Ideal cuando conocemos exactamente la versión que deseamos recuperar.



In [0]:
## 💡 RESTAURAR LA TABLA A LA VERSIÓN 0

### APLICAMOS RESTORE TABLE
spark.sql("""
          RESTORE TABLE catalog_databricks_2026_de.schema_databricks_2026_de.customers_delta_table
          TO VERSION AS OF 0;
          """)

## =========================================================

### VERIFICAMOS LA TABLA
display(
    spark.sql("SELECT * FROM catalog_databricks_2026_de.schema_databricks_2026_de.customers_delta_table")
)
#### ➡️ Resultado: Se restauró a la Versión 0 (Versión donde la tabla fue creada y no existían datos)

## =========================================================

### VERIFICAMOS EL HISTORIAL DE LA TABLA
display(
    spark.sql("DESCRIBE HISTORY catalog_databricks_2026_de.schema_databricks_2026_de.customers_delta_table")
)
#### ➡️ Resultado: Se generó una nueva versión gracias al RESTORE TABLE (Operation: RESTORE)

### 🕒 Restauración por Fecha y Hora (Timestamp)

Permite restaurar la tabla al estado que tenía en un momento específico del tiempo.

```sql
RESTORE TABLE catalog.schema.table
TO TIMESTAMP AS OF 'fecha_hora';
```

🎯 Ideal cuando conocemos el momento en que ocurrió un cambio no deseado, pero no la versión asociada.

---

In [0]:
## 💡 RESTAURAR LA TABLA A LA FECHA Y HORA: 2026-06-11T04:41:15.000+00:00

### APLICAMOS RESTORE TABLE
spark.sql("""
          RESTORE TABLE catalog_databricks_2026_de.schema_databricks_2026_de.customers_delta_table
          TO TIMESTAMP AS OF '2026-06-11T04:41:15.000+00:00';
          """)

## =========================================================

### VERIFICAMOS LA TABLA
display(
    spark.sql("SELECT * FROM catalog_databricks_2026_de.schema_databricks_2026_de.customers_delta_table")
)
#### ➡️ Resultado: Se restauró al TimeStamp 2026-06-11T04:41:15.000+00:00 (Versión donde la tabla tenía un solo registro)

## =========================================================

### VERIFICAMOS EL HISTORIAL DE LA TABLA
display(
    spark.sql("DESCRIBE HISTORY catalog_databricks_2026_de.schema_databricks_2026_de.customers_delta_table")
)
#### ➡️ Resultado: Se generó una nueva versión gracias al RESTORE TABLE (Operation: RESTORE)